In [ ]:
!pip3 install pandas

In [1]:
import os
import sys

print("Current directory:", os.getcwd())
print("Files:", os.listdir())

Current directory: /Users/karthikeyan/Documents/news_popularity_system/Notebook
Files: ['main_pipeline.ipynb']


In [2]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root:", project_root)

Project root: /Users/karthikeyan/Documents/news_popularity_system


In [4]:
import sys
!{sys.executable} -m pip install transformers torch

  Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
  Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl (73.6 MB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
  Using cached pyyaml-6.0.3-cp39-cp39-macosx_11_0_arm64.whl (174 kB)
  Using cached filelock-3.19.1-py3-none-any.whl (15 kB)
     |████████████████████████████████| 80 kB 796 kB/s eta 0:00:01
  Using cached regex-2026.1.15-cp39-cp39-macosx_11_0_arm64.whl (288 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl (3.0 MB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl (447 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached fsspec-2025.10.0-py3-none-any.whl (200 kB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
     |████████████████████████████████| 3.9 MB 3.0 MB/s eta 0:00:01
  Using cached mpmath-1.3.0-py3-none-any.whl (5

In [5]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

/Users/karthikeyan/Documents/news_popularity_system/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/karthikeyan/Documents/news_popularity_system/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers: 4.57.6
PyTorch: 2.8.0


In [7]:
import sys

!{sys.executable} -m pip install nltk textstat

  Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
  Using cached textstat-0.7.13-py3-none-any.whl (177 kB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Using cached pyphen-0.17.2-py3-none-any.whl (2.1 MB)
You should consider upgrading via the '/Users/karthikeyan/Documents/news_popularity_system/.venv/bin/python -m pip install --upgrade pip' command.


In [8]:
import nltk

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/karthikeyan/nltk_data...


True

In [9]:
from nltk.sentiment import SentimentIntensityAnalyzer
import textstat

sia = SentimentIntensityAnalyzer()

print("NLTK and TextStat working!")

NLTK and TextStat working!


In [11]:


from src.preprocessing.text_cleaner import clean_text
from src.model.transformer_encoder import get_embedding
from src.features.popularity_features import *
from src.scoring.popularity_score import calculate_popularity
from src.ranking.article_ranker import rank_articles

In [12]:
# Create models folder
os.makedirs("models", exist_ok=True)                                                                            

In [13]:
import pandas as pd

df = pd.read_csv("/Users/karthikeyan/Documents/news_popularity_system/data/News_dataset.csv")

print(df.shape)

df.head()

df.info()

(216900, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216900 entries, 0 to 216899
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Title        216900 non-null  object
 1   Description  216885 non-null  object
dtypes: object(2)
memory usage: 3.3+ MB


In [14]:
df["Combined_text"] = df["Title"] + "   " + df["Description"]

df.head()


,Title,Description,Combined_text
0,Obama Lays Wreath at Arlington National Cemetery,Obama Lays Wreath at Arlington National Cemete...,Obama Lays Wreath at Arlington National Cemete...
1,A Look at the Health of the Chinese Economy,"Tim Haywood, investment director business-unit...",A Look at the Health of the Chinese Economy ...
2,Nouriel Roubini: Global Economy Not Back to 2008,"Nouriel Roubini, NYU professor and chairman at...",Nouriel Roubini: Global Economy Not Back to 20...
3,Finland GDP Expands In Q4,Finland's economy expanded marginally in the t...,Finland GDP Expands In Q4 Finland's economy ...
4,"Tourism, govt spending buoys Thai economy in J...",Tourism and public spending continued to boost...,"Tourism, govt spending buoys Thai economy in J..."


In [15]:
# Remove rows with missing Description
df = df.dropna(subset=["Description"])

# Reset index after dropping
df = df.reset_index(drop=True)

df.shape

(216885, 3)

In [16]:
df.sample(5000)

,Title,Description,Combined_text
77292,Jeremy Corbyn described Israeli politicians as...,Jeremy Corbyn described unspecified Israeli po...,Jeremy Corbyn described Israeli politicians as...
70035,Obama to Bernie supporters: Don't let disillus...,"One of them will be President Barack Obama, wh...",Obama to Bernie supporters: Don't let disillus...
93376,"Haas, Dent win openers at Madrid Masters",Tommy Haas of Germany and Taylor Dent of the U...,"Haas, Dent win openers at Madrid Masters Tom..."
113375,Bryant's Attorneys Ask for Dismissal,Attorneys for NBA star Kobe Bryant have asked ...,Bryant's Attorneys Ask for Dismissal Attorne...
124699,"McDonald #39;s raises dividend 38pc, shares climb",LOS ANGELES: McDonald #39;s Corp yesterday rai...,"McDonald #39;s raises dividend 38pc, shares cl..."
...,...,...,...
168769,Peru beauty crowned Miss World 2004,Peru #39;s 20-year-old Maria Julia Mantilla Ga...,Peru beauty crowned Miss World 2004 Peru #39...
162744,Vauxhall Plant Set to Face Parts Shortage,Vauxhall car workers were today told to check ...,Vauxhall Plant Set to Face Parts Shortage Va...
66868,Climate-Driven Water Scarcity Could Hit Econom...,"High and Dry: Climate Change, Water and the Ec...",Climate-Driven Water Scarcity Could Hit Econom...
38976,Microsoft's Binning Last-Gen Windows Store App...,Microsoft has announced quite the cull of its ...,Microsoft's Binning Last-Gen Windows Store App...


In [18]:
# Combine Title and Description

df["text"] = (
    df["Title"].fillna("").astype(str)
    + " "
    + df["Description"].fillna("").astype(str)
)

print(df[["Title", "Description", "text"]].head())

                                               Title  \
0   Obama Lays Wreath at Arlington National Cemetery   
1        A Look at the Health of the Chinese Economy   
2   Nouriel Roubini: Global Economy Not Back to 2008   
3                          Finland GDP Expands In Q4   
4  Tourism, govt spending buoys Thai economy in J...   

                                         Description  \
0  Obama Lays Wreath at Arlington National Cemete...   
1  Tim Haywood, investment director business-unit...   
2  Nouriel Roubini, NYU professor and chairman at...   
3  Finland's economy expanded marginally in the t...   
4  Tourism and public spending continued to boost...   

                                                text  
0  Obama Lays Wreath at Arlington National Cemete...  
1  A Look at the Health of the Chinese Economy Ti...  
2  Nouriel Roubini: Global Economy Not Back to 20...  
3  Finland GDP Expands In Q4 Finland's economy ex...  
4  Tourism, govt spending buoys Thai economy in J..

In [19]:
# Clean text
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
# Generate embeddings opt hvy stp
# df["embedding"] = df["clean_text"].apply(get_embedding)

In [22]:
# Feature engineering
df["emotion"] = df["clean_text"].apply(emotion_score)
df["lexical"] = df["clean_text"].apply(lexical_diversity)
df["readability"] = df["clean_text"].apply(readability_score)
df["urgency"] = df["clean_text"].apply(urgency_score)


In [23]:

# Popularity score
df["popularity_score"] = df.apply(
    lambda row: calculate_popularity(
        row["emotion"],
        row["urgency"],
        row["lexical"],
        row["readability"]
    ),
    axis=1
)

In [24]:
# Normalize
df["popularity_score"] = (df["popularity_score"] / df["popularity_score"].max()) * 100

In [25]:
# Ranking
df = rank_articles(df)

In [26]:

# Save
df.to_csv("models/news_popularity_scored.csv", index=False)


In [27]:

# Output
df[["Title","popularity_score"]].head()

,Title,popularity_score
119840,Top of 3rd,100.000000
124690,Wild re-sign D Schultz,85.161801
98457,Pay Up for Growth,79.022461
126605,Should Your Next Car Be New or Used?,78.520635
147422,The Fewest Shares You Can Buy,77.963528
